<a href="https://colab.research.google.com/github/JOSHNA-97886/projects/blob/main/Hangman_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

class HangmanEnv:
    def __init__(self, words):
        self.words = words
        self.reset()

    def reset(self):
        """Resets the environment and selects a new word."""
        self.word = random.choice(self.words).upper()
        self.guessed_word = ["_"] * len(self.word)
        self.attempts_left = 6  # Number of wrong guesses allowed
        self.guessed_letters = set()
        return self.get_state()

    def get_state(self):
        """Returns the current state as a tuple (partially guessed word, attempts left)."""
        return "".join(self.guessed_word), self.attempts_left

    def step(self, action):
        """Takes a step in the environment based on the agent's action (guess)."""
        action = action.upper()
        reward = 0

        if action in self.guessed_letters:
            return self.get_state(), -1, False  # Penalize repeated guesses

        self.guessed_letters.add(action)

        if action in self.word:
            reward = 1  # Reward for a correct guess
            for i, letter in enumerate(self.word):
                if letter == action:
                    self.guessed_word[i] = action
            if "_" not in self.guessed_word:
                return self.get_state(), 10, True  # Big reward for winning
        else:
            self.attempts_left -= 1
            reward = -1  # Penalize incorrect guess

        done = self.attempts_left == 0
        return self.get_state(), reward, done


In [ ]:
import numpy as np

class QLearningAgent:
    def __init__(self, actions, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.q_table = {}  # Q-table (state-action values)
        self.actions = actions  # Possible actions (A-Z)
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate

    def get_q_value(self, state, action):
        """Returns the Q-value for a given state-action pair."""
        return self.q_table.get((state, action), 0)

    def choose_action(self, state):
        """Chooses an action using an epsilon-greedy strategy."""
        if np.random.rand() < self.epsilon:
            return random.choice(self.actions)  # Explore (random action)
        else:
            # Exploit (best known action)
            q_values = {action: self.get_q_value(state, action) for action in self.actions}
            return max(q_values, key=q_values.get)

    def update_q_value(self, state, action, reward, next_state):
        """Updates the Q-table using the Q-learning formula."""
        best_next_action = max(self.actions, key=lambda a: self.get_q_value(next_state, a))
        q_target = reward + self.gamma * self.get_q_value(next_state, best_next_action)
        q_update = (1 - self.alpha) * self.get_q_value(state, action) + self.alpha * q_target
        self.q_table[(state, action)] = q_update


In [ ]:
words = ["python", "hangman", "reinforcement", "learning"]
env = HangmanEnv(words)
agent = QLearningAgent(actions=[chr(i) for i in range(65, 91)])  # A-Z

episodes = 1000  # Training iterations

for episode in range(episodes):
    state = env.reset()
    done = False

    while not done:
        action = agent.choose_action(state[0])  # Select action based on Q-table
        next_state, reward, done = env.step(action)
        agent.update_q_value(state[0], action, reward, next_state[0])
        state = next_state


In [ ]:
state = env.reset()
done = False
print("Word to Guess:", "".join(state[0]))

while not done:
    action = agent.choose_action(state[0])
    next_state, reward, done = env.step(action)
    print(f"Guess: {action} | State: {next_state} | Reward: {reward}")
    state = next_state

if "_" not in state[0]:
    print("Agent WON!")
else:
    print("Agent LOST!")


Word to Guess: ________
Guess: A | State: ('__A_____', 6) | Reward: 1
Guess: E | State: ('_EA_____', 6) | Reward: 1
Guess: G | State: ('_EA____G', 6) | Reward: 1
Guess: R | State: ('_EAR___G', 6) | Reward: 1
Guess: I | State: ('_EAR_I_G', 6) | Reward: 1
Guess: N | State: ('_EARNING', 6) | Reward: 1
Guess: O | State: ('_EARNING', 5) | Reward: -1
Guess: L | State: ('LEARNING', 5) | Reward: 10
Agent WON!
